# 03——建模结果:表格和数字来自一个匹配的运行

从文件中重现出每个** 模型来源** 表格和图
[`02_modelling.ipynb`](02_modelling.ipynb)写给`results/{run_tag}/`——没有示范对象 .
需要。每个度量的笔记本都装入**最佳运行模式**
(`MODEL_DIRS[metric] = None`,以`pf.resolve_model_dir`解决)或**特定型号。
文件夹**您指向它。无模型输出( 人口表、 人口表)
分布，观测到的地图，稀释验证)来自[`01_cohort_stats.ipynb`](01_cohort_stats.ipynb).

** 已执行的笔记本是报告**:每个表格和数字都是内含的，而且
没有任何内容写入磁盘(除了可选的全曲随机截取，即
在 `CONFIG` 中设置 `SAVE_PAPER_ASSETS = True` 以额外写入
出版物资产(`.tex`表，300 -dpi PNG+SVG+PDF图)到`results/paper/`.

| 产出 | 函数 | 资产(保存时) |
| :--- | :--- | :--- |
| 成果表(主文+补充:阶段赢家，全网格，OLS参数) | `paper_tables.generate_all_tables` | `results_{metric}.tex` , `supp_*.tex` , `{metric}_full_grid.tex` , `{metric}_ols_params.tex` |
| 全速随机截取所选模型(由"准确度"数字需要). | `pf.extend_random_effects_to_cohort` | `<model folder>/random_effects.parquet` |
| 无图：{量} _residuals,无图：{量} _qq | `pf.make_model_residuals_plot` , `pf.make_random_intercepts_plot` | `{metric}_residuals` , `{metric}_qq` |
| 无花果：lat_bin_fits,无花果：countries_prediction_fit | `pf.make_combined_seasonal_fit_lat_bin / _country` | `rhr_hrv_combined_seasonal_prediction_fit_*` |
| 无图： demographic_fit | `pf.make_joint_demographics_calibration` | `demographic_fit` |
| 无图： global_residuals_grid | `pf.make_world_contour_residuals` | `world_contour_combined_residuals` |

表格中描述了竞选的全部竞争(每个候选人加上REML).
; 数字使用 `CONFIG` 中选择的模型文件夹。组合 RHR + HRV
数字在两个参数配置时运行。2 按型号比较两个运行模式
和3部分浏览每个模型的诊断。

世界残基轮廓需要`data/README.md`中描述的自然地球文件
`data/extra_data/` .

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import gc
import os
import sys
import time

from IPython.display import HTML, Markdown, SVG, display
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(
    0, os.path.abspath('../src')
)  # 图书馆模块(公开发布)
import file_io as fio
import paper_tables
import publication_figures as pf
import seasonality_lib as sl

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
METRICS = ('rhr', 'hrv')  # 处理的衡量标准(综合数字需要两者)
# 每个公尺的成绩/子文件夹。无=结果/{度量},否则
# 02_modelling.ipynb(rhr_rerun,hrv_rerun)的默认运行标记。
RUN_TAGS = {'rhr': None, 'hrv': None}
# 每公尺:无=跑步成绩最佳的模型(BIC胜者，REML 重制);
# 或模型文件夹名称在结果/{{run_tag}/(或绝对路径)中。
MODEL_DIRS = {'rhr': None, 'hrv': None}
# 还将出版物资产(.tex、png/svg/pdf)写入成果/文件/?
# 否则这个笔记本就是报告
SAVE_PAPER_ASSETS = False
EXTEND_BLUPS = (
    True  # 所选型号的全速随机拦截(见下文)
)
RANDOM_STATE = 42

# 运行其赢家文件夹已保持全速随机截取，并且是
# 从未写入(无默认)。
PROTECTED_RUN_TAGS = ()

data_dir = fio.get_data_dir()
extra_dir = fio.get_extra_dir(data_dir)
results_root = os.path.join(data_dir, fio.RESULTS_SUBDIR)
out_dir = fio.get_paper_dir(data_dir) if SAVE_PAPER_ASSETS else None
if out_dir:
  fio.make_dirs(out_dir)
pf.setup_nature_style()


def show_table(label, latex):
  """Renders a generated LaTeX table inline (preview + collapsible source)."""
  preview = paper_tables.table_preview(latex)
  display(Markdown(f'**`{label}`** — {preview.attrs.get("caption", "")}'))
  display(HTML(preview.to_html(index=False)))
  if preview.attrs.get('notes'):
    display(Markdown(f'<small>{preview.attrs["notes"]}</small>'))
  display(
      HTML(
          '<details><summary>LaTeX source</summary><pre style="font-size:80%">'
          + latex.replace('&', '&amp;').replace('<', '&lt;')
          + '</pre></details>'
      )
  )


def resolve_run_tag(metric, tag):
  if tag is not None:
    return tag
  for candidate in (metric, f'{metric}_rerun'):
    if fio.file_exists(fio.get_results_dir(candidate, data_dir)):
      return candidate
  raise FileNotFoundError(
      f'no results/{metric} or results/{metric}_rerun under {data_dir}: '
      'run 02_modelling.ipynb first or set RUN_TAGS'
  )


RUN_TAGS = {m: resolve_run_tag(m, RUN_TAGS.get(m)) for m in METRICS}
run_dirs = {m: fio.get_results_dir(RUN_TAGS[m], data_dir) for m in METRICS}
model_dirs = {
    m: pf.resolve_model_dir(data_dir, m, RUN_TAGS[m], MODEL_DIRS[m])
    for m in METRICS
}
for m in METRICS:
  print(
      f'{m.upper()}: run {RUN_TAGS[m]!r}\n  model folder:'
      f' {os.path.basename(model_dirs[m])}'
  )
print('publication assets ->', out_dir or 'not saved (SAVE_PAPER_ASSETS=False)')

## 1. 按计量的产出

In [ ]:
for metric in METRICS:
  run_tag, run_dir, model_dir = (
      RUN_TAGS[metric],
      run_dirs[metric],
      model_dirs[metric],
  )
  print(f'\n{"=" * 70}\n{metric.upper()}  ({run_tag})\n{"=" * 70}')

  # 阶段性跑步获奖者(每阶段最低BIC) ->主文本+补充表格。
  results_df = paper_tables.load_model_results(data_dir, metric, run_tag)
  try:
    stage_winners = paper_tables.infer_stage_winners(results_df)
  except KeyError as e:  # 部分运行(MAX_STAGE< 6)缺乏一些阶段
    print(f'Tables skipped: {e}')
  else:
    print('Stage winners:', stage_winners)
    tables = paper_tables.generate_all_tables(
        data_dir,
        out_dir=out_dir,
        metrics=(metric,),
        results_subdirs={metric: run_tag},
        winners={metric: stage_winners},
    )
    for label, latex in tables.items():
      show_table(label, latex)

  # EVERY 干净组群(经验型-贝耶)用户的随机拦截 BLUPs
  # 02_modelling只存储截取器，用于
  # 配置子样本； 绘图随机用户下的精确模式数字。跳过
  # PROTECTED_RUN_TAGS 和为早期扩展的文件夹。
  if EXTEND_BLUPS and run_tag not in PROTECTED_RUN_TAGS:
    t0 = time.time()
    pf.extend_random_effects_to_cohort(
        data_dir, run_dir, metric, model_dir=model_dir
    )
    print(f'  ({time.time() - t0:.0f}s)')
    gc.collect()

  # fig:{meteric} _residuals 和 fig:{meteric} _qq 被选中的模型。
  pf.make_model_residuals_plot(
      data_dir,
      results_root,
      metric,
      out_dir,
      subdir=run_tag,
      model_dir=model_dir,
  )
  pf.make_random_intercepts_plot(
      results_root, metric, out_dir, subdir=run_tag, model_dir=model_dir
  )
  plt.show()
  plt.close('all')

### 合并 RHR + HRV 数字

每个制作者重新装入两个度量的组别数据(模量捆绑，一次一个度量),
重建模型特性，并用所选模型的系数和随机预测
拦截。

In [ ]:
if set(METRICS) >= {'rhr', 'hrv'}:
  kw = dict(rhr_subdir=RUN_TAGS['rhr'], hrv_subdir=RUN_TAGS['hrv'])
  pf.make_combined_seasonal_fit_lat_bin(
      data_dir, results_root, out_dir, model_dirs=model_dirs, **kw
  )  # 无图： lat_bin_fits
  pf.make_combined_seasonal_fit_country(
      data_dir, results_root, out_dir, model_dirs=model_dirs, **kw
  )  # 无图： countries_prediction_fit
  pf.make_joint_demographics_calibration(
      data_dir, results_root, out_dir, model_dirs=model_dirs, **kw
  )  # 无图： demographic_fit
  # 图示：global_residuals_grid(最低的-BIC行)在extra_data中需要天然地球地质学。
  # (见data/README.md);无法使用时跳过。
  if fio.file_exists(os.path.join(extra_dir, pf.WORLD_GEOJSON_FILE)):
    pf.make_world_contour_residuals(
        data_dir, results_root, extra_dir, out_dir, **kw
    )
  else:
    print(
        f'{pf.WORLD_GEOJSON_FILE} not found in {extra_dir}: world contour'
        ' figure skipped'
    )
  plt.show()
  plt.close('all')
  if out_dir:
    print('Combined figures written to', out_dir)
else:
  print('Combined figures need both metrics in METRICS.')

## 2. 比较两个运行(可选)

使用此选项来检查可复制性，例如，重运行时对同一度量的较早运行。
模型在**公式**(对白空间不敏感)上匹配，所以重新命名的阶段仍然
。差异是`run - baseline`;胜负系数按下列单位比较：
基线 标准错误。

In [ ]:
COMPARE_METRIC = 'rhr'
COMPARE_RUN_TAG = RUN_TAGS[COMPARE_METRIC]  # e.g. ' rhr_rerun '
COMPARE_BASELINE_TAG = COMPARE_METRIC  # 基线 运行同一度量衡

FIT_COLS = [
    'BIC',
    'AIC',
    'log_likelihood',
    'marginal_R2',
    'conditional_R2',
    'ICC',
    'rmse',
    'aggregate_R_country',
    'aggregate_R_lat_bin',
    'aggregate_R_age_group',
]
# 高效表格可能带有统计模型摘要标题;使两者都正常化
# 排版到下面使用的 snake_case 名称。
COEF_COLS = {
    'Coef.': 'coef',
    'Std.Err.': 'std_err',
    'P>|z|': 'P_z',
    '[0.025': 'ci_lower',
    '0.975]': 'ci_upper',
}


def _read_csv(path, **kwargs):
  with fio.open_file(path, 'r') as f:
    return pd.read_csv(f, **kwargs)


def list_model_dirs(run_dir):
  """Per-model folders of a run (those containing model-row.csv), sorted by name."""
  return sorted(
      d
      for d in fio.list_dir(run_dir)
      if fio.file_exists(os.path.join(run_dir, d, 'model-row.csv'))
  )


def load_run_table(run_dir, metric):
  path = sl.results_csv_path(run_dir, metric)
  if fio.file_exists(path):
    df = _read_csv(path)
  else:
    df = pd.concat(
        [
            _read_csv(os.path.join(run_dir, d, 'model-row.csv')).assign(
                model_dir=d
            )
            for d in list_model_dirs(run_dir)
        ],
        ignore_index=True,
    )
  df['formula_key'] = df['formula'].astype(str).str.replace(' ', '')
  return df.drop_duplicates('formula_key')


def compare_runs(run_dir, baseline_dir, metric, cols=FIT_COLS):
  a, b = load_run_table(run_dir, metric), load_run_table(baseline_dir, metric)
  cols = [c for c in cols if c in a.columns and c in b.columns]
  merged = a[['model', 'formula_key'] + cols].merge(
      b[['model', 'formula_key'] + cols],
      on='formula_key',
      suffixes=('', '_base'),
      how='outer',
  )
  for c in cols:
    merged[f'd_{c}'] = merged[c] - merged[f'{c}_base']
  order = ['model', 'model_base'] + [
      x for c in cols for x in (c, f'{c}_base', f'd_{c}')
  ]
  return merged[order].sort_values('BIC')


def load_coefs(run_dir, metric):
  path = sl.coefficients_csv_path(run_dir, metric)
  return (
      _read_csv(path, index_col=0).rename(columns=COEF_COLS)
      if fio.file_exists(path)
      else None
  )


cmp_run_dir = fio.get_results_dir(COMPARE_RUN_TAG, data_dir)
cmp_base_dir = fio.get_results_dir(COMPARE_BASELINE_TAG, data_dir)
if COMPARE_RUN_TAG == COMPARE_BASELINE_TAG:
  print(
      'Set COMPARE_RUN_TAG and COMPARE_BASELINE_TAG to two different runs to'
      ' compare them.'
  )
elif fio.file_exists(cmp_run_dir) and fio.file_exists(cmp_base_dir):
  comparison = compare_runs(cmp_run_dir, cmp_base_dir, COMPARE_METRIC)
  display(comparison)
  print(
      'Lowest-BIC model  run / baseline:',
      comparison.iloc[0]['model'],
      '/',
      comparison.sort_values('BIC_base').iloc[0]['model_base'],
  )
  ca, cb = load_coefs(cmp_run_dir, COMPARE_METRIC), load_coefs(
      cmp_base_dir, COMPARE_METRIC
  )
  if ca is not None and cb is not None:
    coef_cmp = ca[['coef', 'std_err']].join(
        cb[['coef', 'std_err']], how='outer', rsuffix='_base'
    )
    coef_cmp['d_coef'] = coef_cmp['coef'] - coef_cmp['coef_base']
    coef_cmp['d_in_se'] = coef_cmp['d_coef'] / coef_cmp['std_err_base']
    display(coef_cmp.round(5))
    print(
        'Max |delta| in baseline SE units:',
        coef_cmp['d_in_se'].abs().max().round(2),
    )

## 3. 浏览所选模型文件夹的诊断

In [ ]:
def show_model_folder(
    model_dir,
    svg_names=(
        'aggregate_validation_lat_bin',
        'aggregate_validation_country',
        'prediction_fit_lat_bin',
        'residuals',
    ),
):
  """Prints the CSV summaries and renders selected SVGs of one model folder."""
  print(f'\n--- {os.path.basename(model_dir)}')
  for name in ('model-row.csv', 'vif.csv'):
    path = os.path.join(model_dir, name)
    if fio.file_exists(path):
      display(_read_csv(path).T if name == 'model-row.csv' else _read_csv(path))
  for name in svg_names:
    path = os.path.join(model_dir, f'{name}.svg')
    if fio.file_exists(path):
      print(f'--- {name}')
      with fio.open_file(path, 'r') as f:
        display(SVG(data=f.read()))


for metric in METRICS:
  show_model_folder(model_dirs[metric])